In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

In [2]:
# ============================================================
# PIPELINE 1 — DATA CLEANING & FEATURE ENGINEERING
# ============================================================

def load_data(path='./df.csv'):
    df = pd.read_csv(path)
    print(f"✅ load_data            : {df.shape}")
    return df

def fix_datetime_cols(df):
    cols = [
        'order_purchase_timestamp',
        'order_approved_at',
        'order_delivered_carrier_date',
        'order_delivered_customer_date',
        'order_estimated_delivery_date'
    ]
    for col in cols:
        df[col] = pd.to_datetime(df[col], errors='coerce')
    print(f"✅ fix_datetime_cols    : done")
    return df

def clean_nulls(df):
    df = df.dropna(subset=[
        'order_approved_at',
        'order_delivered_carrier_date',
        'payment_value'
    ])
    dim_cols = [
        'product_name_lenght', 'product_description_lenght',
        'product_photos_qty',  'product_length_cm',
        'product_height_cm',   'product_width_cm'
    ]
    for col in dim_cols:
        df[col] = df[col].fillna(df[col].median())
    df['payment_sequential']   = df['payment_sequential'].fillna(0)
    df['payment_installments'] = df['payment_installments'].fillna(0)
    df['payment_type']         = df['payment_type'].fillna('unknown')
    print(f"✅ clean_nulls          : {df.shape} | nulls: {df.isnull().sum().sum()}")
    return df

def add_financial_features(df):
    df['revenue']       = df['price'] + df['freight_value']
    df['cost']          = df['price'] * 0.8
    df['profit']        = df['revenue'] - df['cost']
    df['freight_ratio'] = df['freight_value'] / df['revenue']
    print(f"✅ add_financial_features: done")
    return df

def add_time_features(df):
    df['year']    = df['order_purchase_timestamp'].dt.year
    df['month']   = df['order_purchase_timestamp'].dt.month
    df['day']     = df['order_purchase_timestamp'].dt.day
    df['weekday'] = df['order_purchase_timestamp'].dt.day_name()
    print(f"✅ add_time_features    : done")
    return df

def add_purchase_month(df):
    df['purchase_month'] = df['order_purchase_timestamp'].dt.to_period('M').dt.to_timestamp()
    print(f"✅ add_purchase_month   : done")
    return df

def add_cohort_features(df):
    df['Cohort_Month'] = (
        df.groupby('customer_unique_id')['purchase_month']
        .transform('min')
    )
    df['Cohort_Index'] = (
        (df['purchase_month'].dt.year  - df['Cohort_Month'].dt.year)  * 12 +
        (df['purchase_month'].dt.month - df['Cohort_Month'].dt.month)
    )
    print(f"✅ add_cohort_features  : done")
    return df

def run_pipeline_1(path='./df.csv'):
    print("=" * 45)
    print("   PIPELINE 1 — CLEANING & FEATURES")
    print("=" * 45)
    df = load_data(path)
    df = fix_datetime_cols(df)
    df = clean_nulls(df)
    df = add_financial_features(df)
    df = add_time_features(df)
    df = add_purchase_month(df)
    df = add_cohort_features(df)
    df.to_csv('./df_final.csv', index=False)
    print(f"✅ df_final.csv saved   : {df.shape}")
    print("=" * 45)
    return df

In [9]:
# ============================================================
# PIPELINE 2 — SUMMARY TABLE GENERATION
# ============================================================

def build_monthly_summary(df):
    m = df.groupby('purchase_month').agg(
        total_revenue   =('revenue', 'sum'),
        total_profit    =('profit',  'sum'),
        total_orders    =('order_id','nunique'),
        total_customers =('customer_unique_id','nunique')
    ).reset_index()

    m['profit_margin_%'] = np.where(
        m['total_revenue'] > 0,
        (m['total_profit'] / m['total_revenue']) * 100, np.nan
    )
    m['AOV'] = np.where(
        m['total_orders'] > 0,
        m['total_revenue'] / m['total_orders'], np.nan
    )
    prev = m['total_revenue'].shift(1)
    threshold = m['total_revenue'].median() * 0.05
    m['MoM_growth_%']      = np.where(prev >= threshold,
        ((m['total_revenue'] - prev) / prev) * 100, np.nan)
    m['MoM_growth_%']      = m['MoM_growth_%'].clip(-200, 300)
    m['Revenue_3M_MA']     = m['total_revenue'].rolling(3).mean()
    m['MoM_growth_3M_avg_%'] = m['MoM_growth_%'].rolling(3).mean()
    m['YoY_growth_%']      = m['total_revenue'].pct_change(12) * 100
    m = m.sort_values('purchase_month')
    print(f"✅ build_monthly_summary  : {m.shape}")
    return m

def build_category_summary(df):
    c = df.groupby('product_category_name').agg(
        total_revenue     =('revenue',      'sum'),
        total_profit      =('profit',       'sum'),
        total_orders      =('order_id',     'nunique'),
        total_freight     =('freight_value','sum')
    ).reset_index()
    c['profit_margin_%']  = (c['total_profit']  / c['total_revenue']) * 100
    c['avg_order_value']  = c['total_revenue']   / c['total_orders']
    c['freight_ratio_avg']= c['total_freight']   / c['total_revenue']
    c = c.drop(columns='total_freight')
    c = c.sort_values('total_revenue', ascending=False)
    print(f"✅ build_category_summary : {c.shape}")
    return c

def build_state_summary(df):
    s = df.groupby('customer_state').agg(
        total_revenue   =('revenue',           'sum'),
        total_profit    =('profit',            'sum'),
        total_orders    =('order_id',          'nunique'),
        total_customers =('customer_unique_id','nunique'),
        total_freight   =('freight_value',     'sum')
    ).reset_index()
    s['profit_margin_%'] = (s['total_profit']  / s['total_revenue']) * 100
    s['AOV']             =  s['total_revenue']  / s['total_orders']
    s['freight_ratio']   =  s['total_freight']  / s['total_revenue']
    s = s.drop(columns='total_freight')
    s = s.sort_values('total_revenue', ascending=False)
    print(f"✅ build_state_summary    : {s.shape}")
    return s

def build_rfm_table(df):
    analysis_date = df['order_purchase_timestamp'].max() + pd.Timedelta(days=1)
    rfm = df.groupby('customer_unique_id').agg(
        Recency  =('order_purchase_timestamp', lambda x: (analysis_date - x.max()).days),
        Frequency=('order_id',   'nunique'),
        Monetary =('revenue',    'sum')
    ).reset_index()
    rfm['R_score'] = pd.qcut(rfm['Recency'],
                              5, labels=[5,4,3,2,1], duplicates='drop')
    rfm['F_score'] = pd.qcut(rfm['Frequency'].rank(method='first'),
                              5, labels=[1,2,3,4,5])
    rfm['M_score'] = pd.qcut(rfm['Monetary'],
                              5, labels=[1,2,3,4,5], duplicates='drop')
    rfm[['R_score','F_score','M_score']] = (
        rfm[['R_score','F_score','M_score']].astype(int)
    )
    rfm['RFM_Score'] = (rfm['R_score'].astype(str) +
                        rfm['F_score'].astype(str) +
                        rfm['M_score'].astype(str))
    print(f"✅ build_rfm_table        : {rfm.shape}")
    return rfm

def build_segment_summary(rfm):
    def segment(row):
        if row['R_score'] >= 4 and row['F_score'] >= 4 and row['M_score'] >= 4:
            return 'Champions'
        elif row['F_score'] >= 4 and row['M_score'] >= 3:
            return 'Loyal'
        elif row['R_score'] <= 2 and row['F_score'] >= 3:
            return 'At Risk'
        else:
            return 'Regular'
    rfm['Segment'] = rfm.apply(segment, axis=1)
    seg = rfm.groupby('Segment').agg(
        Customer_Count=('customer_unique_id','count'),
        Total_Revenue =('Monetary',          'sum'),
        Avg_Frequency =('Frequency',         'mean')
    ).reset_index()
    print(f"✅ build_segment_summary  : {seg.shape}")
    print(seg)
    return seg, rfm

def save_all_tables(monthly, category, state, segment):
    monthly.to_csv('./monthly_performance_summary.csv',  index=False, float_format='%.2f')
    category.to_csv('./category_performance_summary.csv',index=False, float_format='%.2f')
    state.to_csv('./state_performance_summary.csv',      index=False, float_format='%.2f')
    segment.to_csv('./segment_summary.csv',              index=False, float_format='%.2f')
     segment.to_csv('./segment_summary.csv',              index=False, float_format='%.2f')
    print(f"✅ All 4 CSVs saved!")

def run_pipeline_2(df):
    print("=" * 45)
    print("   PIPELINE 2 — TABLE GENERATION")
    print("=" * 45)
    monthly   = build_monthly_summary(df)
    category  = build_category_summary(df)
    state     = build_state_summary(df)
    rfm       = build_rfm_table(df)
    seg, rfm  = build_segment_summary(rfm)
    save_all_tables(monthly, category, state, seg)
    print("=" * 45)
    return monthly, category, state, rfm, seg

In [10]:
# ============================================================
# RUN EVERYTHING
# ============================================================

# Pipeline 1
df = run_pipeline_1(path='./df.csv')

# Pipeline 2
monthly, category, state, rfm, segment = run_pipeline_2(df)

   PIPELINE 1 — CLEANING & FEATURES
✅ load_data            : (115030, 33)
✅ fix_datetime_cols    : done
✅ clean_nulls          : (115011, 33) | nulls: 0
✅ add_financial_features: done
✅ add_time_features    : done
✅ add_purchase_month   : done
✅ add_cohort_features  : done
✅ df_final.csv saved   : (115011, 44)
   PIPELINE 2 — TABLE GENERATION
✅ build_monthly_summary  : (22, 11)
✅ build_category_summary : (74, 7)
✅ build_state_summary    : (27, 8)
✅ build_rfm_table        : (93335, 8)
✅ build_segment_summary  : (4, 4)
     Segment  Customer_Count  Total_Revenue  Avg_Frequency
0    At Risk           13295     1594408.72       1.006017
1  Champions            6473     2116124.48       1.180751
2      Loyal           16440     3757412.94       1.107421
3    Regular           57127     8642545.42       1.001803
✅ All 4 CSVs saved!
